# Gradient Boosting — Greedy Function Approximation: A Gradient Boosting Machine (2001)

_Papers · Classical ML_

**Build a model as a sum of small trees, each fit to the negative gradient (the residual) of the loss.**

---

This notebook builds gradient boosting from scratch, one step at a time: a hand-worked round, a from-scratch fit loop, a check against scikit-learn, and ablations. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — NumPy

Gradient boosting builds a model as a sum of small trees, each fit to the **negative gradient** (for squared loss, the residual) of the current model. We build it in four steps: (1) one round by hand, (2) a from-scratch fit/predict loop on toy data, (3) a check against scikit-learn, (4) staged error and learning-rate / tree-count ablations.

### Step 1 — One boosting round by hand

Start the model at the mean (Algorithm 1, line 1). The residual `y - F0` is the negative gradient of squared loss (line 3). Fit a depth-1 stump to those residuals, then add it with $\nu = 1$ (Eq. 36). Watch the sum-of-squared-errors fall from 200 to 50 in a single round.

In [ ]:
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor

# Sanity-check the worked example: ONE boosting round, stump, nu = 1.
xw = np.array([[1.], [2.], [3.]])
yw = np.array([10., 20., 30.])
F0w = yw.mean()                                   # init = mean (Alg.1 line 1) -> 20
rw  = yw - F0w                                    # residual = -gradient (line 3) -> [-10, 0, 10]
stump = DecisionTreeRegressor(max_depth=1, random_state=0).fit(xw, rw)
h1  = stump.predict(xw)                           # fit stump to residuals -> [-10, 5, 5] (split x<1.5)
F1w = F0w + 1.0 * h1                              # Eq.36, nu=1 -> [10, 25, 25]
print("worked example  F0 =", F0w, " residuals =", rw.tolist(),
      " h1 =", h1.tolist(), " F1 =", F1w.tolist())
sse_before = float((rw**2).sum())
sse_after  = float(((yw - F1w)**2).sum())
print("  SSE before =", sse_before, " after =", sse_after)  # 200 -> 50

### Step 2 — From-scratch boosting on toy 1-D data

Now the full loop on a smooth 1-D target. `fit_gb` starts at the mean, then for each round computes the residual `y - F` (the negative gradient of squared loss), fits a tree to it, and adds a shrunk copy `lr * tree` to the running model (Eq. 36). `predict_gb` replays the same additive sum on new inputs.

In [ ]:
# Toy 1-D regression target.
rng = np.random.RandomState(0)
X = np.linspace(-3, 3, 60).reshape(-1, 1)
y = np.sin(X).ravel() + 0.3 * X.ravel() + rng.normal(0, 0.05, 60)

LR, N_TREES, DEPTH = 0.1, 100, 3

# Gradient boosting FROM SCRATCH (LS_Boost: squared loss).
def fit_gb(X, y, lr, n_trees, depth):
    F0 = y.mean()                                 # Alg.1 line 1: const minimizing squared loss
    F  = np.full_like(y, F0)
    trees = []
    for m in range(n_trees):
        r = y - F                                 # line 3: -gradient of squared loss = residual
        t = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(X, r)  # line 4
        F = F + lr * t.predict(X)                 # Eq.36: F_m = F_{m-1} + nu * h_m
        trees.append(t)
    return F0, trees

def predict_gb(F0, trees, lr, Xq):
    out = np.full(Xq.shape[0], F0)
    for t in trees:
        out += lr * t.predict(Xq)
    return out

F0, trees = fit_gb(X, y, LR, N_TREES, DEPTH)
mine = predict_gb(F0, trees, LR, X)
print("fitted", len(trees), "trees; first 5 predictions:", np.round(mine[:5], 3))

### Step 3 — Verify against scikit-learn

If our loop really *is* gradient boosting, it should match `GradientBoostingRegressor` with the same hyperparameters to floating-point precision. We compare the two prediction vectors element-wise.

In [ ]:
# VERIFY against sklearn's GradientBoostingRegressor (the oracle).
sk = GradientBoostingRegressor(n_estimators=N_TREES, learning_rate=LR,
                               max_depth=DEPTH, random_state=0).fit(X, y)
skpred = sk.predict(X)
max_diff = float(np.max(np.abs(mine - skpred)))
print("max |mine - sklearn| =", max_diff)
print("np.allclose(mine, sklearn) =", np.allclose(mine, skpred, atol=1e-6))   # -> True
# max |mine - sklearn| ~ 6.7e-16  ->  my from-scratch loop IS GradientBoostingRegressor.

### Step 4 — Staged error and ablations

`staged_mse(k)` rebuilds the prediction from only the first `k` trees, so we can watch train error fall as trees accumulate. Then we ablate two knobs: the learning rate `nu` at a fixed tree budget, and the number of trees at fixed `nu`. Small `nu` under-trains at a fixed budget (Friedman's Table 1); more rounds lower error.

In [ ]:
# Staged additive fit: train MSE as trees accumulate.
def staged_mse(k):
    out = np.full_like(y, F0)
    for t in trees[:k]:
        out += LR * t.predict(X)
    return float(np.mean((out - y) ** 2))

staged = [round(staged_mse(k), 4) for k in (1, 5, 20, 50, 100)]
print("staged train MSE @ [1,5,20,50,100] trees:", staged)
# -> [1.1138, 0.4875, 0.0237, 0.0004, 0.0001]  (error falls fast, then flattens)

# Ablation: learning rate and number of trees.
def final_mse(lr, n):
    F0a, ts = fit_gb(X, y, lr, n, DEPTH)
    pred = predict_gb(F0a, ts, lr, X)
    return round(float(np.mean((pred - y) ** 2)), 5)

print("\nablation lr @100 trees:",
      {nu: final_mse(nu, 100) for nu in (1.0, 0.3, 0.1, 0.03)})
# -> {1.0: 0.0, 0.3: 0.0, 0.1: 5e-05, 0.03: 0.00437}  (small nu under-trains at fixed M -> Table 1)
print("ablation #trees @lr=0.1:",
      {n: final_mse(0.1, n) for n in (1, 5, 20, 100)})
# -> {1: 1.11381, 5: 0.48746, 20: 0.02375, 100: 5e-05}  (more rounds -> lower error)

## Visualize the data & results

_As trees accumulate, does the staged additive model close in on the target, and how fast does train error fall?_

### Step 1 — Rebuild the fit and snapshot the staged model

This cell re-creates the toy data and runs the from-scratch loop, saving snapshots of the running model `F` after 1, 5, 20, and 100 trees. These snapshots let us watch the additive fit close in on the target as rounds accumulate.

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor

# From-scratch LS_Boost; reproduces the staged additive fit on toy 1-D data.
rng = np.random.RandomState(0)
X = np.linspace(-3, 3, 60).reshape(-1, 1)
y = np.sin(X).ravel() + 0.3 * X.ravel() + rng.normal(0, 0.05, 60)
LR, DEPTH = 0.1, 3

F0 = y.mean()
F = np.full_like(y, F0)
trees = []
snap = {}
for m in range(1, 101):
    r = y - F                                              # -gradient of squared loss (line 3)
    t = DecisionTreeRegressor(max_depth=DEPTH, random_state=0).fit(X, r)
    F = F + LR * t.predict(X)                              # Eq.36
    trees.append(t)
    if m in (1, 5, 20, 100):
        snap[m] = F.copy()                                # staged additive snapshots
print("saved snapshots at trees:", sorted(snap))

### Step 2 — Print the staged fit and the MSE curve

We subsample the points for a readable table, then print the target alongside each snapshot `F_m`. The early model is a near-flat nudge off the mean; by 100 trees it hugs the target. The MSE curve falls from ~1.11 to ~1e-4. (Our run, not the paper's numbers.)

In [ ]:
idx = np.linspace(0, 59, 18).astype(int)                  # subsample for the chart
print("x     :", [round(float(v), 3) for v in X.ravel()[idx]])
print("target:", [round(float(v), 3) for v in y[idx]])
for m in (1, 5, 20, 100):
    print(f"F_{m:<3d}:", [round(float(v), 3) for v in snap[m][idx]])

def staged_mse(k):
    out = np.full_like(y, F0)
    for t in trees[:k]:
        out += LR * t.predict(X)
    return round(float(np.mean((out - y) ** 2)), 4)

print("MSE curve:", [[k, staged_mse(k)] for k in (1, 2, 3, 5, 8, 12, 18, 25, 35, 50, 65, 80, 100)])
# F_1 ~ flat nudge off the mean; F_100 hugs the target. MSE: 1.11 -> ~1e-4. Our run, not the paper's.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The learning-rate ablation. Hold the number of trees fixed at 100 and shrink the learning
            rate $\nu$ from $1.0$ to $0.03$. On the smooth 1-D fit, what happens to the final train
            error, and how does that line up with Friedman's Table 1?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Run the from-scratch loop with $M=100$ at $\nu \in \{1.0, 0.3, 0.1, 0.03\}$, changing only $\nu$. — _An honest ablation moves one knob; any difference in final error is attributable to the learning rate._
- Read the final train MSE at each $\nu$: at $\nu=1.0$ and $0.3$ it is essentially $0$ (the 100 trees over-fit the 60 points); at $\nu=0.1$ it is ~5e-5; at $\nu=0.03$ it is the largest (~4e-3) because 100 small steps have not finished descending. — _Small $\nu$ means each step corrects only a little, so at a fixed budget the model is under-trained &mdash; it needs more rounds._
- Connect to Table 1: Friedman shows the best $M$ grows as $\nu$ falls ($\nu=1\to M\!\approx\!15$, $\nu=0.06\to M\!\approx\!326$). — _Same trade-off: shrinking $\nu$ buys generalization but spends more trees; at a fixed $M$, very small $\nu$ leaves error on the table._

**Answer:** At a fixed budget of 100 trees, the largest $\nu$ drives train error to ~0 (over-fitting the
                 60 points), while the smallest $\nu=0.03$ leaves the most error because 100 tiny steps have
                 not finished descending. This is exactly Friedman's $\nu$-$M$ trade-off (Table 1): smaller
                 $\nu$ needs a larger $M$. On held-out data the moderate $\nu$ (with enough trees)
                 generalizes best &mdash; which is why he uses $\nu=0.1$ throughout. The CODEVIZ panel shows
                 the staged fit at $\nu=0.1$ closing the gap as trees accumulate.

</details>

**Problem 2.** Your worked example had $x=[1,2,3]$, $y=[10,20,30]$, $F_0=20$, residuals $[-10,0,10]$, stump
            $h_1=[-10,5,5]$, giving $F_1=[10,25,25]$. Now do the second round (still $\nu=1$, depth-1
            stump). What residuals does it fit, and what is the new SSE?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Residuals after round 1: $\tilde y = y - F_1 = [10-10,\;20-25,\;30-25] = [0,\,-5,\,+5]$. — _Line 3 again: the next tree fits the negative gradient at the current model $F_1$, not $F_0$._
- Fit a depth-1 stump to $[0,-5,5]$. Best split $x\lt 2.5$ ($\{1,2\}$ vs $\{3\}$): left leaf $=\tfrac{0+(-5)}{2}=-2.5$, right leaf $=+5$. So $h_2=[-2.5,-2.5,5]$. — _Each leaf outputs the mean residual on its side; the stump minimizes squared error over the three candidate splits._
- Update $F_2 = F_1 + h_2 = [10-2.5,\;25-2.5,\;25+5] = [7.5,\,22.5,\,30]$. New residuals $[2.5,-2.5,0]$, SSE $=2.5^2+2.5^2+0 = 12.5$. — _SSE fell $200 \to 50 \to 12.5$ &mdash; each round shrinks the leftover error._

**Answer:** Round 2 fits the residuals $[0,-5,5]$ left by round 1. The stump splits at $x\lt 2.5$ giving
                 $h_2=[-2.5,-2.5,5]$, so $F_2=[7.5,22.5,30]$ and the new residuals are $[2.5,-2.5,0]$. The SSE
                 drops from $50$ to $12.5$ &mdash; the same residual-chasing descent, one more step down.

</details>

**Problem 3.** A teammate replaces squared-error loss with absolute-error loss $L(y,F)=|y-F|$ in the
            same boosting skeleton. What single line of the algorithm changes, and to what?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Locate line 3 of ALGORITHM 1 &mdash; the pseudo-response. Everything else (fit a tree, shrink, add) is loss-agnostic. — _The whole point of the paper is that ONE skeleton serves any differentiable loss; only the gradient differs._
- Differentiate $L=|y-F|$: $\partial L/\partial F = -\operatorname{sign}(y-F)$, so the negative gradient is $\tilde y_i = \operatorname{sign}(y_i - F_{m-1}(\mathbf{x}_i))$ (the paper's Eq. 13). — _For absolute error the tree is fit to the sign of the residual, $\pm 1$, not the residual itself &mdash; that is LAD_TreeBoost (&sect;4.2)._
- Note the leaf values also change (a weighted median, Eq. 14) rather than a mean, but the loop is otherwise identical. — _Friedman: insert the new pseudo-response into ALGORITHM 1 to get the LAD boosting method._

**Answer:** Only line 3 (the pseudo-response) changes: for absolute-error loss the negative
                 gradient is $\tilde y_i = \operatorname{sign}(y_i - F_{m-1}(\mathbf{x}_i))$ (Eq. 13), so the
                 tree is fit to the sign of the residual instead of the residual. The fit-tree, shrink,
                 add steps are unchanged. That loss-agnostic skeleton &mdash; swap the gradient, keep the loop
                 &mdash; is the paper's central contribution.

</details>